In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import isnan, when, count, col, translate, expr, length, to_date
import pyspark.sql.functions as F


def drop_nulls(pyspark_df, column_name):
    pyspark_df = pyspark_df.na.drop(subset=[column_name])
    return pyspark_df

def get_duplicates_and_rank(pyspark_df, partition_column, column_to_order, result_column):
    windowSpec = Window.partitionBy(partition_column).orderBy(F.col(column_to_order).desc())
    pyspark_df = pyspark_df.withColumn(result_column, F.row_number().over(windowSpec))
    return pyspark_df

def trim_string_columns(pyspark_df):
    pyspark_df = pyspark_df.select([F.trim(F.col(c)).alias(c) for c in pyspark_df.columns])
    return pyspark_df

def set_na_values(pyspark_df, default_value, list_of_columns):
    pyspark_df = pyspark_df.fillna(value=default_value, subset=list_of_columns)
    return pyspark_df

def categorize_string_values(pyspark_df, value_to_replace, list_of_columns):
    pyspark_df = pyspark_df.replace(value_to_replace, subset = list_of_columns)
    return pyspark_df

df_crm_customer_info = spark.read.table("sales_sports_pyspark.00_bronze_pyspark.crm_cust_info")
df_crm_customer_info = drop_nulls(df_crm_customer_info, "cst_id")
df_crm_customer_info = get_duplicates_and_rank(df_crm_customer_info, "cst_id", "cst_create_date", "flag_last")
df_crm_customer_info = df_crm_customer_info.filter(F.col("flag_last") == 1).drop("flag_last")
df_crm_customer_info = trim_string_columns(df_crm_customer_info)
df_crm_customer_info = set_na_values(df_crm_customer_info, "N/A", ["cst_marital_status", "cst_gndr"])
df_crm_customer_info = categorize_string_values(df_crm_customer_info, {"M": "Married", "m": "Married", "S": "Single", "s": "Single"}, ["cst_marital_status"])
df_crm_customer_info = categorize_string_values(df_crm_customer_info, {"F": "Female", "f": "Female", "M": "Male", "m": "Male"}, ["cst_gndr"])
df_crm_customer_info = df_crm_customer_info.withColumn("cst_create_date", F.to_date("cst_create_date"))

df_crm_customer_info.write.format("delta").mode("overwrite").saveAsTable("sales_sports_pyspark.01_silver_pyspark.crm_customer_info")


In [0]:

df_crm_product_info = spark.read.table("sales_sports_pyspark.00_bronze_pyspark.crm_product_info")
df_crm_product_info = drop_nulls(df_crm_product_info, "prd_id")
df_crm_product_info = trim_string_columns(df_crm_product_info)
df_crm_product_info = df_crm_product_info.withColumn("cat_id", F.substring("prd_key", 1, 5))
df_crm_product_info = df_crm_product_info.withColumn("cat_id", translate("cat_id", "-.", "_,"))
df_crm_product_info = df_crm_product_info.withColumn("prd_key", expr('substring(prd_key, 7,length(prd_key))'))
df_crm_product_info = set_na_values(df_crm_product_info, 0, ["prd_cost", "prd_line"])
df_crm_product_info = categorize_string_values(df_crm_product_info, {"M": "Mountain", "m": "Mountain", "R": "Road", "r": "Road", "S": "Other Sales", "s": "Other Sales", "T": "Touring", "t": "Touring"}, ["prd_line"])
windowSpec = Window.partitionBy("prd_key").orderBy(F.col("prd_start_dt"))
df_crm_product_info = df_crm_product_info.withColumn("prd_end_dt", F.lead("prd_start_dt").over(windowSpec))
df_crm_product_info = df_crm_product_info.withColumn("prd_end_dt", F.date_add(F.col("prd_end_dt"), -1))

df_crm_product_info = df_crm_product_info.withColumn("prd_start_dt", F.to_date("prd_start_dt"))
df_crm_product_info.write.format("delta").mode("overwrite").saveAsTable("sales_sports_pyspark.01_silver_pyspark.crm_product_info")

df_crm_product_info.show()

+------+----------+--------------------+--------+--------+------------+----------+------+
|prd_id|   prd_key|              prd_nm|prd_cost|prd_line|prd_start_dt|prd_end_dt|cat_id|
+------+----------+--------------------+--------+--------+------------+----------+------+
|   601|   BB-7421|   LL Bottom Bracket|      24|    NULL|  2013-07-01|      NULL| CO_BB|
|   602|   BB-8107|   ML Bottom Bracket|      45|    NULL|  2013-07-01|      NULL| CO_BB|
|   603|   BB-9108|   HL Bottom Bracket|      54|    NULL|  2013-07-01|      NULL| CO_BB|
|   478|   BC-M005|Mountain Bottle Cage|       4|Mountain|  2013-07-01|      NULL| AC_BC|
|   479|   BC-R205|    Road Bottle Cage|       3|    Road|  2013-07-01|      NULL| AC_BC|
|   596|BK-M18B-40|Mountain-500 Blac...|     295|Mountain|  2013-07-01|      NULL| BI_MB|
|   597|BK-M18B-42|Mountain-500 Blac...|     295|Mountain|  2013-07-01|      NULL| BI_MB|
|   598|BK-M18B-44|Mountain-500 Blac...|     295|Mountain|  2013-07-01|      NULL| BI_MB|
|   599|BK

In [0]:
df_crm_sales_details = spark.read.table("sales_sports_pyspark.00_bronze_pyspark.crm_sales_details")
df_crm_sales_details = trim_string_columns(df_crm_sales_details)
#df_crm_sales_details = df_crm_sales_details.filter(df_crm_sales_details.sls_order_dt < 19000101)

df_crm_sales_details = df_crm_sales_details.withColumn("sls_order_dt", F.when((F.col("sls_order_dt") == "0") | (F.length(F.col("sls_order_dt")) != 8), None)
     .otherwise(F.col("sls_order_dt"))
)

df_crm_sales_details = df_crm_sales_details.withColumn("sls_due_dt", F.when((F.col("sls_due_dt") == "0") | (F.length(F.col("sls_due_dt")) != 8), None)
     .otherwise(F.col("sls_due_dt"))
)

df_crm_sales_details = df_crm_sales_details.withColumn("sls_ship_dt", F.when((F.col("sls_ship_dt") == "0") | (F.length(F.col("sls_ship_dt")) != 8), None)
     .otherwise(F.col("sls_ship_dt"))
)

df_crm_sales_details = df_crm_sales_details.withColumn("sls_order_dt", to_date("sls_order_dt", "yyyyMMdd"))
df_crm_sales_details = df_crm_sales_details.withColumn("sls_due_dt", to_date("sls_due_dt", "yyyyMMdd"))
df_crm_sales_details = df_crm_sales_details.withColumn("sls_ship_dt", to_date("sls_ship_dt", "yyyyMMdd"))

df_crm_sales_details = df_crm_sales_details.withColumn("sls_quantity", col("sls_quantity").cast("int"))
df_crm_sales_details = df_crm_sales_details.withColumn("sls_price", col("sls_price").cast("int"))
df_crm_sales_details = df_crm_sales_details.withColumn("sls_sales", col("sls_sales").cast("int"))

df_crm_sales_details = df_crm_sales_details.withColumn(
    "sls_sales",
    F.when(
        (F.col("sls_sales").isNull()) |
        (F.col("sls_sales") <= 0) |
        (F.col("sls_sales") != (F.col("sls_quantity") * F.abs(F.col("sls_price")))),
        F.col("sls_quantity") * F.col("sls_price")
    ).otherwise(F.col("sls_sales")) # Alternative value if conditions are not met
)

df_crm_sales_details = df_crm_sales_details.withColumn(
    "sls_price",
    F.when(
        (F.col("sls_price").isNull()) |
        (F.col("sls_price") <= 0),
        F.col("sls_sales") /  F.nullifzero(F.col("sls_quantity"))
    ).otherwise(F.col("sls_sales")) # Alternative value if conditions are not met
)

df_crm_sales_details.write.format("delta").mode("overwrite").saveAsTable("sales_sports_pyspark.01_silver_pyspark.crm_sales_details")


In [0]:
df_erp_cust_az12 = spark.read.table("sales_sports_pyspark.00_bronze_pyspark.erp_cust_az12")

df_erp_cust_az12 = df_erp_cust_az12.withColumn("cid", F.regexp_replace("cid", "^NAS", ""))

df_erp_cust_az12 = df_erp_cust_az12.withColumn(
    "bdate", 
    F.when(F.col("bdate") > F.current_date(), F.lit(None))
     .otherwise(F.col("bdate"))
)

df_erp_cust_az12 = trim_string_columns(df_erp_cust_az12)
# Clean GEN column: F -> Female, M -> Male, null/empty -> N/A
df_erp_cust_az12 = df_erp_cust_az12.withColumn(
    "gen",
    F.when((F.col("gen").isNull()) | (F.col("gen") == ""), "N/A")
     .when((F.col("gen") == "F") | (F.col("gen") == "f") | (F.col("gen") == "FEMALE"), "Female")
     .when((F.col("gen") == "M") | (F.col("gen") == "m") | (F.col("gen") == "MALE"), "Male")
     .otherwise(F.col("gen"))
)

df_erp_cust_az12 = df_erp_cust_az12.withColumn("bdate", F.to_date("bdate"))
df_erp_cust_az12.write.format("delta").mode("overwrite").saveAsTable("sales_sports_pyspark.01_silver_pyspark.erp_cust_az12")

In [0]:
df_loc_a101 = spark.read.table("sales_sports_pyspark.00_bronze_pyspark.erp_loc_a101")
df_loc_a101 = trim_string_columns(df_loc_a101)
df_loc_a101 = df_loc_a101.withColumn("cid", translate("cid", "-.", ""))

df_loc_a101 = df_loc_a101.withColumn(
    "cntry",
    F.when((F.col("cntry").isNull()) | (F.col("cntry") == ""), "N/A")
     .when(F.col("cntry") == "DE", "Germany")
     .when((F.col("cntry") == "US") | (F.col("cntry") == "USA"), "United States")
     .otherwise(F.col("cntry"))
)

df_loc_a101.write.format("delta").mode("overwrite").saveAsTable("sales_sports_pyspark.01_silver_pyspark.erp_loc_a101")

In [0]:
df_px_cat_g1v2 = spark.read.table("sales_sports_pyspark.00_bronze_pyspark.erp_px_cat_g1v2")
df_px_cat_g1v2 = trim_string_columns(df_px_cat_g1v2)

df_px_cat_g1v2.write.format("delta").mode("overwrite").saveAsTable("sales_sports_pyspark.01_silver_pyspark.erp_px_cat_g1v2")